In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import warnings
warnings.filterwarnings("ignore")

builder = (
    SparkSession.builder
    .appName("delta-minio-test")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        ",".join([
            "io.delta:delta-spark_2.12:3.2.0",
            "org.apache.hadoop:hadoop-aws:3.3.4",
            "com.amazonaws:aws-java-sdk-bundle:1.12.262",
        ])
    )
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ebc9965a-9c38-4a40-83de-ea08bbfd2d26;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 91ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0

In [2]:
!spark-submit \
  --packages io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262 \
  --conf "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension" \
  --conf "spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog" \
  /workspace/rltm_bi_pltfrm/jobs/silver_to_gold/gold_bars_1m.py

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ba68ffbb-d491-41f5-98b7-7b050315b04b;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 147ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 f

In [3]:
gold_path = "s3a://lakehouse/gold/gold_price_bars_1m"

df = spark.read.format("delta").load(gold_path)
print("row_count =", df.count())
df.orderBy("bar_start_ts", ascending=False).show(20, truncate=False)

26/03/21 23:28:41 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/03/21 23:28:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


row_count = 9
+------+------------+--------+-------------------+-------------------+----------+-----------+-----------+-----------+-----------+-----------+----------+
|symbol|source_name |currency|bar_start_ts       |bar_end_ts         |trade_date|open_price |high_price |low_price  |close_price|avg_price  |tick_count|
+------+------------+--------+-------------------+-------------------+----------+-----------+-----------+-----------+-----------+-----------+----------+
|XAU   |gold_api_com|USD     |2026-03-21 17:31:00|2026-03-21 17:32:00|2026-03-21|4492.200195|4492.200195|4492.200195|4492.200195|4492.200195|1         |
|XAU   |gold_api_com|USD     |2026-03-21 17:18:00|2026-03-21 17:19:00|2026-03-21|4492.200195|4492.200195|4492.200195|4492.200195|4492.200195|1         |
|XAU   |gold_api_com|USD     |2026-03-21 17:17:00|2026-03-21 17:18:00|2026-03-21|4492.200195|4492.200195|4492.200195|4492.200195|4492.200195|1         |
|XAU   |gold_api_com|USD     |2026-03-21 17:15:00|2026-03-21 17:16:0